# 00 — Fermi-LAT data products: a quick tour

**Goal of this notebook (~10 min).** Before we run any tool, look at the *raw* data product you start from: the FT1 photon list and the FT2 spacecraft history. We will:

1. Read the FT1 files with `astropy.io.fits`.
2. Plot the energy spectrum and the ROI sky distribution.
3. Decode the `EVENT_CLASS` / `EVENT_TYPE` bitfields the LAT processing pipeline writes.
4. Look at one row of FT2 to understand pointing / livetime.
5. Run `gtselect` and `gtmktime` by hand to produce a clean event list.

This is the **Level-2** product (cf. [FSSC, *Cicerone — LAT Data Products*](https://fermi.gsfc.nasa.gov/ssc/data/analysis/documentation/Cicerone/Cicerone_Data/LAT_DP.html)). Everything `fermipy` and `gammapy` do downstream is bookkeeping on top of these two files plus the IRFs and diffuse templates.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.coordinates import SkyCoord
import astropy.units as u
from glob import glob
from pathlib import Path

# Local data directory (already populated by hand from the LAT Data Server,
# Meyer ISAPP 2021 query: PG 1553+113, 20 deg, MET 239557417 -> 271093418).
ANADIR = Path(os.environ.get("LECTURE_DATA", "/opt/lecture-data")) / "pg1553"

FT1_GLOB = str(ANADIR / "L*_PH0?.fits")
FT2_GLOB = str(ANADIR / "L*_SC00.fits")

print("ANADIR    :", ANADIR)
print("FT1 files:", sorted(p.name for p in ANADIR.glob("L*_PH0?.fits")))
print("FT2 file  :", sorted(p.name for p in ANADIR.glob("L*_SC00.fits")))

## 1. The FT1 photon list

Each row is *one* gamma-ray candidate. The LAT Data Server returns multiple FT1 chunks per query (the server splits long time ranges to keep individual files manageable), so we have three `*_PH00.fits`, `*_PH01.fits`, `*_PH02.fits` chunks covering the full 1-yr window MET 239 557 417 → 271 093 418 (2008-08-04 → 2009-08-04 UTC).

The columns are documented in [FSSC, *Cicerone — LAT Data Columns*](https://fermi.gsfc.nasa.gov/ssc/data/analysis/documentation/Cicerone/Cicerone_Data/LAT_Data_Columns.html). The most-used ones, with the FSSC definitions verbatim:

| column | unit | FSSC definition |
|---|---|---|
| `TIME`         | s    | "Mission elapsed time when the event was detected" — MET = total seconds since 00:00:00 UTC on 2001-01-01 |
| `ENERGY`       | MeV  | "Reconstructed energy of the event" |
| `RA`           | deg  | "Reconstructed direction of the event in Right Ascension" |
| `DEC`          | deg  | "Reconstructed direction of the event in Declination" |
| `L`, `B`       | deg  | reconstructed direction in galactic coordinates |
| `THETA`        | deg  | "Reconstructed angle of incidence of the event with respect to the LAT boresight (+Z axis of the spacecraft)" |
| `ZENITH_ANGLE` | deg  | "Angle between the reconstructed event direction and the zenith line (originates at the centre of the Earth and passes through the centre of mass of the spacecraft)" |
| `EVENT_CLASS`  | bits | "A bitfield indicating which event-class selections a given event has passed" |
| `EVENT_TYPE`   | bits | "A bitfield indicating which event-type selections a given event has passed" |

The MET reference epoch is encoded in the FT1 header as `MJDREFI = 51910` (= 2001-01-01) and `MJDREFF ≈ 7.43e-4` d (a small offset that pins the reference to the chosen TIMESYS, here `TT`).

In [ ]:
ft1_files = sorted(glob(FT1_GLOB))
assert ft1_files, f"no FT1 files matched {FT1_GLOB}"

# Inspect the first chunk of data 
hdul = fits.open(ft1_files[0])
hdul.info()

events_chunk0 = hdul["EVENTS"].data
hdr           = hdul["EVENTS"].header

print("\nEVENTS columns + units:")
for c in events_chunk0.columns:
    print(f"  {c.name:22s} fmt={c.format:6s} unit={c.unit or '-'}")

print(f"\nrows in this chunk: {len(events_chunk0):,}")
print(f"TSTART = {hdr['TSTART']:.0f} MET s")
print(f"TSTOP  = {hdr['TSTOP']:.0f} MET s   "
      f"(chunk spans {(hdr['TSTOP']-hdr['TSTART'])/86400:.1f} d)")
print(f"TIMESYS = {hdr['TIMESYS']}, MJDREFI = {hdr['MJDREFI']}, "
      f"MJDREFF = {hdr['MJDREFF']:.6e}")

The first chunk alone has ~232 k events — *a lot* of photons for one source — because the FT1 file contains *every* event reconstructed inside the 20°-radius search cone, *across all event classes*. The spatial / class / energy / zenith cuts that select a SOURCE-class point-source dataset are applied later by `gtselect`.

For the energy and sky plots below we concatenate all three chunks so we see the whole 1-yr window at once.

In [ ]:
all_events = np.concatenate([fits.open(f)["EVENTS"].data for f in ft1_files])
print(f"total events across {len(ft1_files)} chunks: {len(all_events):,}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# energy spectrum (counts per log-E bin)
e = all_events["ENERGY"]
ax[0].hist(np.log10(e), bins=60)
ax[0].set_xlabel(r"$\log_{10}(E\,/\,\mathrm{MeV})$")
ax[0].set_ylabel("counts per bin")
ax[0].set_title("FT1 reconstructed energy (all events, no cuts)")

# sky distribution in galactic coords
c = SkyCoord(all_events["RA"]*u.deg, all_events["DEC"]*u.deg).galactic
ax[1].hexbin(c.l.wrap_at(180*u.deg).deg, c.b.deg,
             gridsize=80, cmap="magma", bins="log")
ax[1].invert_xaxis()                         # convention: l increases to the left
ax[1].set_xlabel("galactic l (deg)")
ax[1].set_ylabel("galactic b (deg)")
ax[1].set_title("FT1 sky distribution (20° cone around PG 1553+113)")

# mark PG 1553+113 itself
pg = SkyCoord(238.929*u.deg, 11.190*u.deg).galactic
ax[1].plot(pg.l.wrap_at(180*u.deg).deg, pg.b.deg,
           marker='+', color='cyan', mew=2, ms=14, label='PG 1553+113')
ax[1].legend(loc='lower right')
fig.tight_layout()

**Things to notice.**

- The energy histogram falls steeply: most photons sit at a few hundred MeV, very few above 100 GeV. The LAT effective area also varies with energy, but the dominant effect is the *source-population* spectrum (∝ E^−2.x for typical extragalactic sources, plus diffuse).
- PG 1553+113 sits at $(l,b)\approx(21.9^\circ,\ +43.9^\circ)$, well off the galactic plane — not much bright diffuse background, not many nearby sources. It's a nice target.
- The 20deg search cone is much larger than the 10deg-side ROI we will fit later. The padding lets the LAT PSF and the diffuse templates "leak" properly into the ROI from outside its boundary, which keeps the fit unbiased near the edges.

## 2. Event class / event type bitfields

The LAT processing pipeline runs *every* photon through a chain of background-rejection cuts and stores the per-event result in two FITS columns of TFORM `32X` — i.e. a 32-bit field, one bit per cut. We **never re-run the cuts**; we *select* events whose mask has a given bit set ([FSSC, *Cicerone — LAT IRFs*](https://fermi.gsfc.nasa.gov/ssc/data/analysis/documentation/Cicerone/Cicerone_LAT_IRFs/IRF_overview.html)).

For Pass 8 P8R3 (the current reprocessing), the standard event-class hierarchy is:

| EVENT_CLASS bit | name | `evclass` value (= `2**bit`) |
|----:|---|----:|
|  4  | P8R3_TRANSIENT020      |    16 |
|  6  | P8R3_TRANSIENT010      |    64 |
|  7  | **P8R3_SOURCE**        |   **128** |
|  8  | P8R3_CLEAN             |   256 |
|  9  | P8R3_ULTRACLEAN        |   512 |
| 10  | P8R3_ULTRACLEANVETO    |  1024 |
| 11  | P8R3_SOURCEVETO        |  2048 |
| 16  | P8R3_TRANSIENT015S     | 65536 |

The classes are **nested**: a photon that passes `ULTRACLEANVETO` automatically passes `ULTRACLEAN`, `CLEAN`, `SOURCE`, `TRANSIENT010`, `TRANSIENT020`. A standard point-source analysis uses `evclass=128` (SOURCE) — strict enough that residual cosmic-ray contamination is small, loose enough to keep good photon statistics.

The event-type bitfield encodes orthogonal *partitions* of the same events:

| EVENT_TYPE bit | name | `evtype` value |
|---:|---|----:|
| 0 | FRONT  (thin-converter)         |   1 |
| 1 | BACK   (thick-converter)        |   2 |
| 2 | PSF0   (worst PSF quartile)     |   4 |
| 3 | PSF1                            |   8 |
| 4 | PSF2                            |  16 |
| 5 | PSF3   (best PSF quartile)      |  32 |
| 6 | EDISP0 (worst E-disp quartile)  |  64 |
| 7 | EDISP1                          | 128 |
| 8 | EDISP2                          | 256 |
| 9 | EDISP3 (best E-disp quartile)   | 512 |

Each event has *exactly one* of FRONT/BACK, *exactly one* of PSF0..3, and *exactly one* of EDISP0..3. `evtype = 3` (`= 1|2`) means "FRONT or BACK", i.e. *all* events independent of conversion type. A multi-component PSF analysis (the one fermipy uses by default for PG 1553) splits events into 4 sub-datasets along the PSF partition and fits them jointly with class-specific IRFs.

**FITS bit-packing convention.** When astropy reads a `32X` column it returns an `(N, 32)` boolean array. The most-significant bit of the underlying 32-bit integer is at *array index 0*, so FSSC bit `n` lives at *array index `(31 − n)`*. The cell below packs the bool array back into a single integer per row using `np.packbits` with big-endian byte order, so `mask & 128` is True iff the SOURCE bit is set.

In [ ]:
def bits_to_mask(bits32):
    """Pack a (N, 32) bool array (FITS 32X) into an N-vector of FSSC bitmasks.

    Bits are stored MSB-first across the 32 bits; FSSC "bit n" maps to
    value 2**n in the returned uint32 mask.
    """
    return np.packbits(bits32.astype(np.uint8), axis=1).view('>u4').reshape(-1)

evclass_mask = bits_to_mask(events_chunk0["EVENT_CLASS"])
evtype_mask  = bits_to_mask(events_chunk0["EVENT_TYPE"])

print(f"first event:  EVENT_CLASS = 0x{evclass_mask[0]:08x}   "
      f"EVENT_TYPE = 0x{evtype_mask[0]:08x}")

CLASSES = {"TRANSIENT020": 16, "TRANSIENT010": 64, "SOURCE": 128,
           "CLEAN": 256, "ULTRACLEAN": 512, "ULTRACLEANVETO": 1024,
           "SOURCEVETO": 2048, "TRANSIENT015S": 65536}
print("\nEVENT_CLASS membership (this chunk only):")
for name, val in CLASSES.items():
    n = ((evclass_mask & val) != 0).sum()
    print(f"  {name:18s} (evclass={val:6d}): {n:8,d}  "
          f"({100*n/len(evclass_mask):5.1f} %)")

TYPES = {"FRONT": 1, "BACK": 2,
         "PSF0": 4, "PSF1": 8, "PSF2": 16, "PSF3": 32,
         "EDISP0": 64, "EDISP1": 128, "EDISP2": 256, "EDISP3": 512}
print("\nEVENT_TYPE membership (this chunk only):")
for name, val in TYPES.items():
    n = ((evtype_mask & val) != 0).sum()
    print(f"  {name:8s} (evtype={val:4d}): {n:8,d}  "
          f"({100*n/len(evtype_mask):5.1f} %)")

**Sanity checks.**

- The four PSF sub-classes partition the dataset exactly: each event has one and only one of PSF0..3 set, so the four percentages should sum to 100 %. Same for EDISP0..3, and FRONT+BACK = 100 %.
- The class hierarchy nests: SOURCE ≥ CLEAN ≥ ULTRACLEAN ≥ ULTRACLEANVETO. The numbers shrink as the class becomes stricter.
- TRANSIENT020 ≥ TRANSIENT010 ≥ SOURCE: the transient classes are deliberately looser (more accepted events, more residual background) so they can be used on short flares where there are many source photons (e.g., for GRBs).

## 3. The FT2 spacecraft file

One row roughly every 30 s, telling you where Fermi was pointing and how good the livetime was. You almost never read it directly — `gtltcube` takes it to build a livetime cube — but it is good to *see* once.

Documented columns ([FSSC, *Cicerone — LAT Data Columns*](https://fermi.gsfc.nasa.gov/ssc/data/analysis/documentation/Cicerone/Cicerone_Data/LAT_Data_Columns.html)):

| column | unit | FSSC definition |
|---|---|---|
| `START`, `STOP`         | s   | "Mission Elapsed Time of start / end of interval" |
| `RA_SCZ`, `DEC_SCZ`     | deg | "viewing direction at START (RA / Dec of S/C +z axis)" |
| `RA_ZENITH`, `DEC_ZENITH` | deg | "RA / Dec of zenith at START" |
| `LIVETIME`              | s   | "Accumulated livetime of the LAT during the interval from START to STOP" |
| `DATA_QUAL`             | flag | run-level quality flag set by the LAT processing pipeline (values below) |
| `LAT_CONFIG`            | flag | "1 = nominal science configuration, 0 = not recommended for analysis" |
| `IN_SAA`                | bool | "Whether the spacecraft was in the SAA at START" |

`LIVETIME < (STOP − START)` whenever the LAT was paused (SAA passages, calibration, downlinks, target-of-opportunity slews, …); the ratio is what `gtltcube` integrates over the sky to produce the exposure map.

In [ ]:
sc_files = sorted(glob(FT2_GLOB))
assert sc_files, f"no FT2 file matching {FT2_GLOB}"
sc_hdul = fits.open(sc_files[0])
sc_hdul.info()

sc = sc_hdul["SC_DATA"].data
print(f"\nFT2 rows: {len(sc):,}  "
      f"(~{len(sc)*30/86400/365.25:.2f} yr at 30 s/row)")
print("first row:")
for n in ("START", "STOP", "RA_SCZ", "DEC_SCZ", "RA_ZENITH", "DEC_ZENITH",
         "LIVETIME", "DATA_QUAL", "LAT_CONFIG", "IN_SAA"):
    if n in sc.columns.names:
        print(f"  {n:12s} = {sc[n][0]}")

# Livetime fraction over the analysis window.
in_win = (sc['START'] >= 239557417) & (sc['STOP'] <= 271093418)
ontime   = (sc['STOP'][in_win] - sc['START'][in_win]).sum()
livetime = sc['LIVETIME'][in_win].sum()
print(f"\nover the 1-yr Meyer window (in-window FT2 rows: {in_win.sum():,}):")
print(f"  on-time  Σ(STOP-START) = {ontime:.3e} s = {ontime/86400:.1f} d")
print(f"  livetime Σ(LIVETIME)   = {livetime:.3e} s = {livetime/86400:.1f} d")
print(f"  livetime fraction       = {livetime/ontime*100:.1f} %")

**On-time vs livetime.** The cell prints `Σ(LIVETIME) / Σ(STOP−START) ≈ 90 %` — i.e. *given that the LAT was on*, it accumulated effective livetime ~90 % of the elapsed seconds; the rest is lost mostly to dead-time. But `Σ(STOP−START)` itself is ~303 d out of the 365.25 d wall-clock window — the missing ~60 d is when the LAT was *off* (mostly SAA crossings, ~13 % of the orbit, plus calibration / downlinks). So the wall-clock duty cycle is closer to ~75 %.

## 4. Plot the data-quality flags vs time

Before any analysis, sanity-check that the LAT was actually taking science-quality data over the time range you queried. The two flags that matter:

`DATA_QUAL` — set by the LAT processing pipeline when each run is finalized (FSSC values):

| value | meaning |
|------:|---|
|  1  | good data |
|  2  | GPS timing anomaly (small clock wobble; *still usable* for non-pulsar work) |
|  0  | bad data |
| −1  | solar-flare contamination |
| −2  | particle event |
| −3  | excluded interval (e.g. GRB 221009A) |

`LAT_CONFIG` — 1 means the LAT was in its nominal science configuration; 0 means engineering / commissioning / off-pointed mode.

The recommended GTI filter for a standard SOURCE-class analysis is

```
(DATA_QUAL>0) && (LAT_CONFIG==1)
```

— this keeps both ordinary good data (`DATA_QUAL==1`) and the GPS-flagged but otherwise fine intervals (`DATA_QUAL==2`), and rejects everything ≤ 0 ([FSSC, *Cicerone — Data Preparation*](https://fermi.gsfc.nasa.gov/ssc/data/analysis/documentation/Cicerone/Cicerone_Data_Exploration/Data_preparation.html)).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
in_win = (sc['START'] >= 239557417) & (sc['STOP'] <= 271093418)
ax.plot(sc["START"][in_win], sc["DATA_QUAL"][in_win],
        ".", ms=2, label="DATA_QUAL")
ax.plot(sc["START"][in_win], sc["LAT_CONFIG"][in_win],
        ".", ms=2, alpha=0.5, label="LAT_CONFIG")
ax.set_xlabel("MET (s)")
ax.set_ylabel("flag value")
ax.set_ylim(-0.2, 2.2)
ax.set_title("Spacecraft-file quality flags over the 1-yr Meyer window")
ax.legend()
fig.tight_layout()

## 5. Filtering by hand: `gtselect` + `gtmktime`

These are the first two boxes of the FSSC analysis flowchart ([FSSC, *Cicerone — Data Preparation*](https://fermi.gsfc.nasa.gov/ssc/data/analysis/documentation/Cicerone/Cicerone_Data_Exploration/Data_preparation.html)). When fermipy's `gta.setup()` runs them under the hood it uses the parameters from `config.yaml`. Knowing what each parameter does is most of fermipy fluency.

- **`gtselect`** keeps events that pass *event-level* cuts:
  - spatial: `ra`, `dec`, `rad` (acceptance cone, deg),
  - temporal: `tmin`, `tmax` (MET; `INDEF` = use `TSTART`/`TSTOP` from the FT1 header),
  - energy: `emin`, `emax` (MeV),
  - zenith: `zmax` (deg),
  - quality: `evclass`, `evtype` (the bits we just decoded).
- **`gtmktime`** builds Good-Time-Intervals from the FT2 file using a logical
  expression of FT2 keywords and ANDs them into the FT1's `GTI` extension:
  - `filter`: the boolean expression (here `(DATA_QUAL>0) && (LAT_CONFIG==1)`),
  - `roicut="no"`: do *not* cut on the ROI staying inside the LAT FoV (we already did the radial cut in `gtselect`; using `roicut="yes"` would also throw out intervals where the source approached within `zmax` of the limb),
  - `apply_filter="yes"`: actually drop events outside the new GTIs.

**Why `zmax = 90°`?** Photons reconstructed at zenith ≥ 113° are inside the Earth's limb and are dominated by atmospheric γ-ray showers. The FSSC SOURCE-class recommendation, `zmax = 90°`, leaves a comfortable ~23° margin against the limb so that residual atmospheric contamination is negligible without throwing away too much exposure.

**Why `evclass=128, evtype=3`?** SOURCE class is the FSSC-recommended default for point-source analyses (looser than CLEAN — keeps statistics — but strict enough that residual cosmic rays sit well below the source). `evtype=3 = 1|2` means "FRONT or BACK", i.e. *all* events regardless of conversion type. Meyer's PG 1553 setup further splits the SOURCE photons into PSF0..3 components inside fermipy.

**Important quirk.** `gtselect` and `gtmktime` both read parameter defaults from a `.par` file in `$PFILES`. Setting them via `gt_apps` overrides the defaults *and* writes the new values back to that local `.par` for the rest of the session — that is why the FSSC manual warns *"once you have selected a value for a parameter, do not modify it throughout the analysis chain"*: re-running an upstream tool with a different value can leave inconsistent header keywords in downstream files.

In [ ]:
# gtselect can take either a single FT1 file or an "@listfile" of paths.
# We have 3 chunks, so build the listfile.
events_txt = ANADIR / "events.txt"
events_txt.write_text("\n".join(str(p) for p in ft1_files) + "\n")
print("events.txt:")
print(events_txt.read_text())

import gt_apps                                  # provided by fermitools
from gt_apps import filter as gtselect
gtselect["infile"]  = f"@{events_txt}"
gtselect["outfile"] = str(ANADIR / "PG1553_filtered.fits")
gtselect["ra"]      = 238.929                   # PG 1553+113
gtselect["dec"]     = 11.190
gtselect["rad"]     = 10                        # ROI radius (deg)
gtselect["tmin"]    = "INDEF"                   # use TSTART from FT1
gtselect["tmax"]    = "INDEF"                   # use TSTOP  from FT1
gtselect["emin"]    = 100                       # MeV (Meyer's lower bound)
gtselect["emax"]    = 1000000                   # 1 TeV
gtselect["zmax"]    = 90                        # deg, FSSC recommendation
gtselect["evclass"] = 128                       # SOURCE
gtselect["evtype"]  = 3                         # FRONT + BACK

print("\ngtselect command:")
print(gtselect.command())
gtselect.run()

In [ ]:
from gt_apps import maketime as gtmktime
gtmktime["evfile"]       = str(ANADIR / "PG1553_filtered.fits")
gtmktime["outfile"]      = str(ANADIR / "PG1553_mktime.fits")
gtmktime["scfile"]       = sc_files[0]
gtmktime["filter"]       = "(DATA_QUAL>0) && (LAT_CONFIG==1)"
gtmktime["roicut"]       = "no"
gtmktime["apply_filter"] = "yes"

print("gtmktime command:")
print(gtmktime.command())
gtmktime.run()

In [ ]:
# What survived?
flt = fits.open(ANADIR / "PG1553_mktime.fits")
flt.info()

n_in  = sum(fits.open(f)["EVENTS"].header["NAXIS2"] for f in ft1_files)
n_sel = fits.open(ANADIR / "PG1553_filtered.fits")["EVENTS"].header["NAXIS2"]
n_out = flt["EVENTS"].header["NAXIS2"]
print(f"\ntotal events before                        : {n_in:>9,}")
print(f"events after gtselect (cone+E+zenith+class): {n_sel:>9,}  "
      f"({100*n_sel/n_in:5.1f} %)")
print(f"events after gtmktime (+ DATA_QUAL & LAT_CONFIG GTIs): {n_out:>9,}  "
      f"({100*n_out/n_in:5.1f} %)")

**Why didn't `gtmktime` remove any events?** With this dataset it usually doesn't, because the LAT Data Server already pre-filtered the FT1 chunks before delivering them — the events you got back are already inside `(DATA_QUAL>0) && (LAT_CONFIG==1)` GTIs. What `gtmktime` *does* still do is *refine the GTI extension* (it splits intervals at every flag transition, so the GTI count goes up — see the `5,463 R` in the new GTI extension above). The event count usually only drops when you process FT1 files exported from the *Data Catalog* directly (no server-side filtering), or when you tighten the filter expression beyond what the server applied.

You just executed the first two boxes of the FSSC flowchart by hand. Notebook 01 picks up at the third box (`gtltcube` → `gtbin` → `gtexpcube2` → `gtsrcmaps` → `gtlike`) and lets fermipy stitch all of it together via `GTAnalysis.setup()`.

